# Token Adapter Pattern via Unified Planning (UP)

This notebook demonstrates how to construct a robust abstraction layer between a simulator that emits non-standard or dialect-specific tokens (like `BlocksworldSimulator` emitting `ontable` and `handempty`) and an LLM Planner that has strong priors for standard PDDL (`on-table` and `arm-empty`).

By using the Unified Planning library, we map raw environment strings into strongly-typed objects and fluents. When we need to generate PDDL for the LLM or for a classical solver, UP compiles the unified object schema into proper syntax automatically, preventing the LLM from entering infinite refinement loops over syntax errors.

In [ ]:
import unified_planning as up
from unified_planning.shortcuts import *
from unified_planning.io.pddl_writer import PDDLWriter

# 1. Define the Standard Environment Schema (What the agent expects)
Block = UserType('block')

clear = Fluent('clear', BoolType(), b=Block)
on_table = Fluent('on-table', BoolType(), b=Block) # Standard PDDL
arm_empty = Fluent('arm-empty', BoolType())        # Standard PDDL
on = Fluent('on', BoolType(), b1=Block, b2=Block)
holding = Fluent('holding', BoolType(), b=Block)

problem = Problem('blocksworld_standard')
problem.add_fluent(clear, default_initial_value=False)
problem.add_fluent(on_table, default_initial_value=False)
problem.add_fluent(arm_empty, default_initial_value=False)
problem.add_fluent(on, default_initial_value=False)
problem.add_fluent(holding, default_initial_value=False)

# Create blocks
blocks = [Object(f'block_{i}', Block) for i in range(3)]
problem.add_objects(blocks)

print("Environment Schema correctly initialized.")

## Environment Token Intake (The Adapter)

Suppose our physical environment or simulator returns a raw state snapshot containing poorly formatted tokens. We define a simple translation map that maps these specific strings to our UP Fluent objects.

In [ ]:
# The raw trace from the Simulator
raw_simulator_state = [
    "ontable(block_0)",
    "ontable(block_1)",
    "handempty()",
    "clear(block_0)",
    "on(block_2, block_1)",
    "clear(block_2)"
]

# The Adapter Map (Environment -> UP Fluent)
token_map = {
    "ontable": on_table,
    "handempty": arm_empty,
    "clear": clear,
    "on": on,
    "holding": holding
}

# We parse the string and apply it securely to the UP problem
import re

for state_str in raw_simulator_state:
    # Simple regex to extract Predicate(arg1, arg2)
    match = re.match(r'([a-zA-Z0-9_-]+)\((.*)\)', state_str)
    if match:
        pred_str = match.group(1)
        args_str = match.group(2)
        
        # Look up the correct Fluent object
        if pred_str in token_map:
            fluent = token_map[pred_str]
            
            if args_str:
                # Map string args to UP Objects
                args = [problem.object(arg.strip()) for arg in args_str.split(',')]
                problem.set_initial_value(fluent(*args), True)
            else:
                # Arity 0 (like handempty)
                problem.set_initial_value(fluent(), True)
                
print("Raw simulator state safely ingested into UP constraint schema.")

## Output to Standard PDDL

Now, when the pipeline needs to construct the `domain.pddl` and `problem.pddl` to feed to the LLM agent, it outputs the *Standard* PDDL generated by the UP Writer. The LLM will no longer see `ontable` or `handempty`, it will natively see the standard PDDL `on-table` and `arm-empty` it was trained on.

In [ ]:
from textwrap import dedent

writer = PDDLWriter(problem)
problem_pddl = writer.get_problem()

print("=== GENERATED PROBLEM PDDL ===")
print(problem_pddl)

print("\nNotice how the generated initial state uses (arm-empty) and (on-table block_0), \"solving\" the dialect issue without LLM hallucination.")